\
# Gauge variation and BRST

`gauge_variation` and `brst_transformation` are odd or even `FieldOperator`s
on a compiled Lagrangian. They use the same covariant-derivative convention
as the rest of the toolkit,
$$
D_\mu = \partial_\mu - i g A_\mu.
$$


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    DC,
    FS,
    Field,
    Gamma,
    GaugeGroup,
    GaugeRepresentation,
    LORENTZ_INDEX,
    Model,
    PartialD,
    dirac_field,
    scalar_field,
)
from lagrangian.operator_action import brst_transformation, gauge_variation
from symbolic.spenso_structures import gauge_generator, structure_constant
from symbolic.tensor_canonicalization import canonize_full
from symbolic.vertex_engine import I


\
## Abelian gauge invariance

For a U(1) charge $q$,
$$
\delta\Phi = +iqg\,\alpha\,\Phi,\qquad
\delta A_\mu = \partial_\mu\alpha.
$$
A charge-conserving Lagrangian canonicalizes to zero; a charged scalar
coupled to a neutral fermion does not.


In [2]:
g_u1, mu, m2, y = S("g"), S("mu"), S("m2"), S("y")
A = Field("A", spin=1, indices=(LORENTZ_INDEX,), self_conjugate=True)
U1 = GaugeGroup(name="U1", abelian=True, coupling=g_u1, charge="Q", gauge_boson=A)
Phi_q = scalar_field("Phi", self_conjugate=False, quantum_numbers={"Q": 1})
psi_q = dirac_field("psi", quantum_numbers={"Q": -1})
psi_neutral = dirac_field("eta", quantum_numbers={"Q": 0})

L_u1 = Model(
    m2 * Phi_q.bar * Phi_q + I * psi_q.bar() * Gamma(mu) * DC(psi_q, mu),
    gauge_groups=(U1,),
).lagrangian()
delta_u1 = gauge_variation(group=U1, parameter="alpha")
dL = L_u1.apply_operator(delta_u1)
show(
    "delta(L) after canonicalization",
    canonize_full(
        dL.to_symbolica(),
        lorentz_indices=(mu,),
        spinor_indices=(S("i_bar_psi_U1"), S("i_psi_U1"), S("i_bar_psi_covd"), S("i_psi_covd")),
    ),
)

L_wrong = Model(y * psi_neutral.bar() * psi_neutral() * Phi_q).lagrangian()
show("charged scalar × neutral Yukawa", L_wrong.apply_operator(delta_u1).to_symbolica().expand())

L_ok = Model(y * psi_q.bar() * psi_q() * Phi_q).lagrangian()
show("charge-conserving Yukawa", L_ok.apply_operator(delta_u1).to_symbolica().expand())


delta(L) after canonicalization
0

charged scalar × neutral Yukawa
-1𝑖*g*y*Phi*alpha*eta(i_decl_1)*etabar(i_decl_1)

charge-conserving Yukawa
-1𝑖*g*y*Phi*alpha*psi(i_decl_1)*psibar(i_decl_1)



\
## Non-abelian variation

The gauge boson transforms as
$$
\delta A^a_\mu = \partial_\mu\alpha^a - g f^{abc}\alpha^b A^c_\mu.
$$
The Yang–Mills term $-\tfrac14 F^2$ is invariant after canonicalization.


In [3]:
gS = S("gS")
G = Field("G", spin=1, indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX), self_conjugate=True)
SU3 = GaugeGroup(
    name="SU3",
    abelian=False,
    coupling=gS,
    gauge_boson=G,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fundamental"),
    ),
)
delta_su3 = gauge_variation(group=SU3, parameter="alpha")

mu_a, aa = S("mu"), S("aa")
show(
    "delta(A^a_mu)",
    Model(G(mu_a, aa), gauge_groups=(SU3,)).lagrangian().apply_operator(delta_su3).to_symbolica().expand(),
)

mu_fs, nu_fs = S("mu_fs"), S("nu_fs")
L_ym = Model(
    -(Expression.num(1) / Expression.num(4)) * FS(SU3, mu_fs, nu_fs, S("a_fs")) * FS(SU3, mu_fs, nu_fs, S("a_fs")),
    gauge_groups=(SU3,),
).lagrangian()
show("L_YM", L_ym.to_symbolica())
show(
    "delta(L_YM) canonicalized",
    canonize_full(
        L_ym.apply_operator(delta_su3).to_symbolica().expand(),
        run_gamma=False,
        run_color=False,
        infer_indices=True,
    ),
)


delta(A^a_mu)
-gS*f(coad(8, aa),coad(8, a_delta_SU3_1),coad(8, a_delta_SU3_2))*alpha(a_delta_SU3_1)*G(mu,a_delta_SU3_2)+PartialD(alpha(aa),mu)

L_YM
-1/4*gS*g(mink(4, mu_fs),mink(4, mu_decl_1))*f(coad(8, a_fs),coad(8, fs_adj_decl_1),coad(8, fs_adj_decl_2))*PartialD(G(nu_fs,a_fs),mu_decl_1)*G(mu_fs,fs_adj_decl_1)*G(nu_fs,fs_adj_decl_2)-1/4*gS*g(mink(4, mu_fs),mink(4, mu_decl_1))*f(coad(8, a_fs),coad(8, fs_adj_decl_3),coad(8, fs_adj_decl_4))*PartialD(G(nu_fs,a_fs),mu_decl_1)*G(mu_fs,fs_adj_decl_3)*G(nu_fs,fs_adj_decl_4)+1/4*gS*g(mink(4, nu_fs),mink(4, mu_decl_1))*f(coad(8, a_fs),coad(8, fs_adj_decl_1),coad(8, fs_adj_decl_2))*PartialD(G(mu_fs,a_fs),mu_decl_1)*G(mu_fs,fs_adj_decl_1)*G(nu_fs,fs_adj_decl_2)+1/4*gS*g(mink(4, nu_fs),mink(4, mu_decl_1))*f(coad(8, a_fs),coad(8, fs_adj_decl_3),coad(8, fs_adj_decl_4))*PartialD(G(mu_fs,a_fs),mu_decl_1)*G(mu_fs,fs_adj_decl_3)*G(nu_fs,fs_adj_decl_4)-1/4*gS^2*f(coad(8, a_fs),coad(8, fs_adj_decl_1),coad(8, fs_adj_decl_2))*f(coad(8, a_fs),coad(8, fs_adj

\
A local mass term $m^2 W_\mu W^\mu$ is not gauge invariant. That is a useful
false-positive check: the variation must *not* canonicalize to zero.


In [4]:
from feynpy import WEAK_ADJ_INDEX, WEAK_FUND_INDEX
from symbolic.spenso_structures import weak_gauge_generator, weak_structure_constant

W = Field("W", spin=1, indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX), self_conjugate=True)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=S("g2"),
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(
        GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet"),
    ),
)
mu_bad, aw_bad = S("mu_bad"), S("aw_bad")
L_mass = Model(
    S("m2") * W(mu_bad, aw_bad) * W(mu_bad, aw_bad),
    gauge_groups=(SU2L,),
    fields=(W,),
).lagrangian()
show(
    "delta(m^2 W W) canonicalized",
    canonize_full(
        L_mass.apply_operator(gauge_variation(group=SU2L, parameter="alpha_bad")).to_symbolica().expand(),
        run_gamma=False,
        run_color=False,
        infer_indices=True,
    ),
)


delta(m^2 W W) canonicalized
2*m2*PartialD(alpha_bad(canon_dummy_5_2),canon_dummy_0_1)*W(canon_dummy_0_1,canon_dummy_5_2)



\
## BRST

With the same sign convention,
$$
s A_\mu^a = D_\mu c^a,\qquad
s c^a = -\tfrac{g}{2} f^{abc} c^b c^c,\qquad
s\bar c^a = B^a,\qquad
s B^a = 0.
$$
Nilpotency $s^2=0$ is checked after canonicalization. Gauge fixing is written
as an exact term $sK$.


In [5]:
c_gh = Field(
    "c",
    spin=0,
    kind="ghost",
    ghost_of=G,
    self_conjugate=False,
    conjugate_symbol=S("cbar"),
    indices=(COLOR_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
Baux = Field("Baux", spin=0, self_conjugate=True, indices=(COLOR_ADJ_INDEX,))
s = brst_transformation(group=SU3, ghost=c_gh, auxiliary=Baux)
a = S("a")

sG = G(mu, a).apply_operator(s)
sc = c_gh(a).apply_operator(s)
scbar = c_gh.bar(a).apply_operator(s)
sB = Baux(a).apply_operator(s)
show("s G", sG.to_symbolica().expand())
show("s c", sc.to_symbolica().expand())
show("s cbar", scbar.to_symbolica().expand())
show("s B", sB.to_symbolica().expand())
show(
    "s^2 G canonicalized",
    canonize_full(sG.apply_operator(s).to_symbolica().expand(), infer_indices=True, field_heads=(G, c_gh, Baux)),
)
show(
    "s^2 c canonicalized",
    canonize_full(sc.apply_operator(s).to_symbolica().expand(), infer_indices=True, field_heads=(c_gh,)),
)


s G
gS*f(coad(8, a),coad(8, a_brst_SU3_1),coad(8, a_brst_SU3_2))*G(mu,a_brst_SU3_1)*c(a_brst_SU3_2)+PartialD(c(a),mu)

s c
-1/2*gS*f(coad(8, a),coad(8, a_brst_SU3_3),coad(8, a_brst_SU3_4))*c(a_brst_SU3_3)*c(a_brst_SU3_4)

s cbar
Baux(a)

s B
0

s^2 G canonicalized
0

s^2 c canonicalized
0



In [6]:
xi = S("xi")
K = Model(
    c_gh.bar(a) * PartialD(G(mu, a), mu) + xi / Expression.num(2) * c_gh.bar(a) * Baux(a),
    gauge_groups=(SU3,),
    fields=(G, c_gh, Baux),
).lagrangian()
sK = K.apply_operator(s)
show("K", K.to_symbolica())
show("s(K)", sK.to_symbolica())
show(
    "s(L_YM + s(K)) canonicalized",
    canonize_full(
        (L_ym + sK).apply_operator(s).to_symbolica().expand(),
        infer_indices=True,
        field_heads=(G, c_gh, Baux),
        run_color=False,
    ),
)

PhiQ = scalar_field("PhiQ", self_conjugate=False, quantum_numbers={"Q": 1})
A_u1 = Field("A_brst", spin=1, self_conjugate=True, indices=(LORENTZ_INDEX,))
cQ = Field(
    "cQ",
    spin=0,
    kind="ghost",
    ghost_of=A_u1,
    self_conjugate=False,
    conjugate_symbol=S("cQbar"),
    quantum_numbers={"GhostNumber": 1},
)
U1Q = GaugeGroup(name="U1Q", abelian=True, coupling=S("eQ"), gauge_boson=A_u1, charge="Q")
s_u1 = brst_transformation(group=U1Q, ghost=cQ, auxiliary=Field("BQ", spin=0, self_conjugate=True))
L_kin = Model(DC(PhiQ.bar, mu) * DC(PhiQ, mu), gauge_groups=(U1Q,)).lagrangian()
show(
    "s((D Phi)^dagger D Phi) canonicalized",
    canonize_full(
        L_kin.apply_operator(s_u1).to_symbolica().expand(),
        infer_indices=True,
        field_heads=(A_u1, cQ, PhiQ),
    ),
)


K
1/2*xi*cbar(a)*Baux(a)+g(mink(4, mu),mink(4, mu_decl_1))*PartialD(G(mu,a),mu_decl_1)*cbar(a)

s(K)
gS*g(mink(4, mu),mink(4, mu_decl_1))*f(coad(8, a),coad(8, a_brst_SU3_15),coad(8, a_brst_SU3_16))*PartialD(c(a_brst_SU3_16),mu_decl_1)*G(mu,a_brst_SU3_15)*cbar(a)+gS*g(mink(4, mu),mink(4, mu_decl_1))*f(coad(8, a),coad(8, a_brst_SU3_15),coad(8, a_brst_SU3_16))*PartialD(G(mu,a_brst_SU3_15),mu_decl_1)*cbar(a)*c(a_brst_SU3_16)+1/2*xi*Baux(a)^2+g(mink(4, mu),mink(4, mu_decl_1))*PartialD(G(mu,a),mu_decl_1)*Baux(a)+g(mink(4, mu),mink(4, mu_decl_1))*PartialD(PartialD(c(a),mu_decl_1),mu)*cbar(a)

s(L_YM + s(K)) canonicalized
0

s((D Phi)^dagger D Phi) canonicalized
0

